<a href="https://colab.research.google.com/github/Rahilralu/pytorch/blob/main/NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
torch.cuda.is_available()


True

In [55]:
x,y = load_breast_cancer(return_X_y=True)
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2)

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [56]:
x_train_scaled

array([[-0.8119871 ,  2.31611251, -0.85913594, ..., -1.75245125,
        -2.15788961, -1.36952234],
       [-0.90169718, -0.89031063, -0.86972309, ..., -0.54126914,
        -0.53085413,  1.28388215],
       [-0.78114926, -0.09278575, -0.81312256, ..., -0.69407331,
        -0.82099543, -0.97056305],
       ...,
       [ 0.94296628,  4.62706912,  0.88082125, ...,  1.38877676,
        -0.09484068,  0.03496612],
       [-1.26978884, -0.82268425, -1.28506508, ..., -0.65438984,
        -0.24872778, -0.13741031],
       [-0.49800183, -1.64352858, -0.52604795, ..., -0.32460652,
        -0.13651843, -0.9000947 ]])

In [57]:
x_train_scaled_tensor = torch.from_numpy(x_train_scaled).float()
x_test_scaled_tensor = torch.from_numpy(x_test_scaled).float()

y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)




In [58]:
train_dataset = TensorDataset(x_train_scaled_tensor,y_train_tensor)

In [59]:
x_train_scaled_tensor.shape

torch.Size([455, 30])

In [60]:
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle=True)

In [61]:
class BCNet(nn.Module):

  def __init__(self):
    super(BCNet,self).__init__()
    self.fc1 = nn.Linear(30,64)
    self.fc2 = nn.Linear(64,32)
    self.fc3 = nn.Linear(32,1)

    self.dropout = nn.Dropout(0.5)

  def forward(self,x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.dropout(x)
    x = F.sigmoid(self.fc3(x))

    return x

In [62]:
model = BCNet()

print(model._modules)

{'fc1': Linear(in_features=30, out_features=64, bias=True), 'fc2': Linear(in_features=64, out_features=32, bias=True), 'fc3': Linear(in_features=32, out_features=1, bias=True), 'dropout': Dropout(p=0.5, inplace=False)}


In [63]:
criterion =  nn.BCELoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [64]:
print(model)

BCNet(
  (fc1): Linear(in_features=30, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [65]:
epochs = 20

for epochs in range(epochs):
  model.train()
  running_loss = 0.0

  for x_batch,y_batch in train_loader:
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = criterion(preds,y_batch)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print(f"Epoch {epochs + 1}: Loss was {running_loss/len(train_loader)} ")

Epoch 1: Loss was 0.644659948348999 
Epoch 2: Loss was 0.49791170756022135 
Epoch 3: Loss was 0.33971457481384276 
Epoch 4: Loss was 0.21464519451061884 
Epoch 5: Loss was 0.16216986229022343 
Epoch 6: Loss was 0.11937401692072551 
Epoch 7: Loss was 0.10762069299817086 
Epoch 8: Loss was 0.093265134592851 
Epoch 9: Loss was 0.08522031406561534 
Epoch 10: Loss was 0.07747502891967693 
Epoch 11: Loss was 0.07480164604882399 
Epoch 12: Loss was 0.07216517562046647 
Epoch 13: Loss was 0.08946025396386782 
Epoch 14: Loss was 0.06266813489298026 
Epoch 15: Loss was 0.061044200261433916 
Epoch 16: Loss was 0.05260346829891205 
Epoch 17: Loss was 0.057316730613820255 
Epoch 18: Loss was 0.05447155007471641 
Epoch 19: Loss was 0.05363572562734286 
Epoch 20: Loss was 0.04569431484366457 


In [66]:
with torch.no_grad():
  model.eval()

  preds = model(x_test_scaled_tensor)
  loss = criterion(preds,y_test_tensor).item()

  accuracy = ((preds >= 0.5) ==  y_test_tensor ).float().mean().item()

In [68]:
accuracy

1.0